In [ ]:
from vistiq.io import ImageWriterConfig, ImageWriter, ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.utils import ArrayIteratorConfig, check_device, resolve_futures
from vistiq.core import Tiler, TilerConfig, Untiler, UntilerConfig, labels_to_masks
from vistiq.preprocess import (
    FuncProcessor,
    FuncProcessorConfig,
    PreprocessFlow,
    PreprocessFlowConfig,
    ResizeConfig,
    Resize,
    RescaleConfig,
    Rescale,
    DoG,
    DoGConfig,
    PreprocessorConfig,
    Preprocessor,
)
from vistiq.segment import (
    RegionFilterConfig,
    RegionFilter,
    RangeFilterConfig,
    RangeFilter,
    RegionAnalyzerConfig,
    RegionAnalyzer,
    MicroSAMSegmenter,
    MicroSAMSegmenterConfig,
    MicroSAMMerger,
    MicroSAMMergerConfig,
    TiledSegmentationFlow,
    TiledSegmentationFlowConfig,
    SegmentationFlow,
    SegmentationFlowConfig,
)
from vistiq.analysis import (
    CoincidenceDetectorConfig,
    CoincidenceDetector,
    AnalysisFlowConfig,
    AnalysisFlow,
    IoSMetricsCalculatorConfig,
    LabelOverlapCalculatorConfig,
    OverlapCalculator,
    metrics_calculator_configs,
    region_map_from_dataframe,
    KnnAnalysis,
    KnnAnalysisConfig,
    RnnAnalysis,
    RnnAnalysisConfig,
    SpatialScopeConfig,
)
from vistiq.matrix import (
    MatrixAggregator,
    MatrixAggregatorConfig,
    MatrixCombiner,
    MatrixCombinerConfig,
    MatrixFormatter,
    MatrixFormatterConfig,
    ValueFilter,
    ValueFilterConfig,
    TopKFilter,
    TopKFilterConfig,
)
from vistiq.matrix.types import UPPER, UPPER_ND
from vistiq.graph import (
    GraphFormatterConfig,
    GraphBuilder,
    GraphBuilderConfig,
    GraphQuery,
    GraphQueryConfig,
    GraphQueryFormatter,
    GraphQueryFormatterConfig,
    HierarchyBuilderConfig,
    load_analysis,
    resolve_subtree_origins,
    save_analysis,
    subtree_origin_key,
)

from prefect import flow, task
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait
from prefect.futures import resolve_futures_to_results
from scipy.ndimage import binary_dilation

import stackview
import os
import copy
import numpy as np
import math
import logging
import pandas as pd
import itertools

from typing import Any, List, Tuple
from pathlib import Path


# Spatial analysis

Runs segmentation + post-hoc homotypic/heterotypic kNN/RNN analysis and saves an analysis bundle for downstream napari animation.


# Configure logger and check availability of accelerators

In [ ]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

# Load image

In [ ]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="../data/Animal 1.lif"; scene_index=0
#path="../../cshl/Taylor_GWAS/high/189/dgrp.189.g1.animal1.lif"; scene_index=1
#path="../../cshl/Taylor_GWAS/high/189/dgrp.189.g1.animal2.lif"; scene_index=0
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [ ]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    #substack="Z:20-50"
)
img, metadata = ImageLoader(ilc).run(path)
metadata

In [ ]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [ ]:
tissue_ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
        
        #FuncProcessorConfig(
        #    func="numpy.sum", 
        #    kwargs={"axis":("Z")}, # Project all channels into one
        #    strict_axis=False,     # don't throw exception if the input is a single channel image already
        #    #dtype=np.uint16,
        #),


    ]
)
#tissue_img, tissue_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
#metadata, tissue_metadata, tissue_img.shape

In [ ]:
#tissue_metadata = copy.deepcopy(tissue_metadata)
#tissue_metadata["channel_names"] = ["Lobe"]
# segment tissue
#tissue_labels = TiledSegmentationFlow(tsfcfg).run(tissue_img, metadata=tissue_metadata, workers=2, verbose=0)


In [ ]:
#stackview.slice(tissue_labels)

# Configuration for 3D Tissue Segmentation

In [ ]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        #RangeFilterConfig(
        #    attribute="volume",
        #    range=(100000, np.inf),
        #),
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(2000, np.inf),
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-xz", 
            range=(2000, np.inf),
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-yz", 
            range=(2000, np.inf),
        ),
        RangeFilterConfig(
            attribute="aspect_ratio", 
            range=(0.5, 1.0),
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.5,
    consensus_threshold=0.75,
)

# Configuration for Region Analysis

In [ ]:
racfg = RegionAnalyzerConfig(
    properties=["slice_annotations","volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe",
    map_axes=True,
)
ra = RegionAnalyzer(racfg)


# Configuration for Cell Segmentation

In [ ]:
# specify preprocessing config for cells
cell_ppcfg = PreprocessFlowConfig(
    processors = [
        #DoGConfig(
        #    sigma_low=1, # 5, 
        #    sigma_high=2, #12, 
        #    normalize=True,
        #    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z focal plane and channel
        #)
    ]
)


In [ ]:
# Specify segmentation config
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
    #gpu_fraction=0.3,
)

min_cell_radius = 2.0
max_cell_radius = 7.0
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(np.pi*min_cell_radius**2, np.pi*max_cell_radius**2)
        )
    ]
)

cell_sfcfg = SegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
)

In [ ]:
@flow
def analyze_cells(labels: list[np.ndarray], metadata: list[dict[str, Any]]) -> list[pd.DataFrame]:
    print ([l.shape for l in labels])
    print ([m["channel_names"] for m in metadata])
    racfg = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe"
    )
    ra = RegionAnalyzer(racfg)

    measurements = ra.run.map(labels, metadata=metadata)

    cdcfg = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    )
    label_index_combinations = list(itertools.combinations(range(len(labels)), 2))
    l1 = [labels[c[0]] for c in label_index_combinations]
    l2 = [labels[c[1]] for c in label_index_combinations]
    sn = [(metadata[c[0]]["channel_names"][0], metadata[c[1]]["channel_names"][0]) for c in label_index_combinations]
    print (sn)
    #for la1, la2, sna in zip(l1,l2,sn): 
    cim = CoincidenceDetector(cdcfg).run.map(l1, l2, stack_names=sn)
    return measurements
 

In [ ]:
lobe_spatial_scope = SpatialScopeConfig(match={"channel": "Lobe"})

knn_spatial_cfg = KnnAnalysisConfig(
    k=5,
    mode="homotypic",
    scope=lobe_spatial_scope,
)
rnn_spatial_cfg = RnnAnalysisConfig(
    radius=15,
    mode="homotypic",
    scope=lobe_spatial_scope,
)

acfg = AnalysisFlowConfig(
    region_analyzer=RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config=ArrayIteratorConfig(slice_def=()),
        output_type="dataframe",
        index_on="object_id",
        map_axes=True,
    ),
    #coincidence_detector = CoincidenceDetectorConfig(
    #    method=IoSMetricsCalculatorConfig(),
    #    iterator_config=ArrayIteratorConfig(slice_def=()),
    #    mode="outline",
    #),
    hierarchy_builder=HierarchyBuilderConfig(
        orphan_strategy="drop",  # "group",
    ),
    overlap_calculator=LabelOverlapCalculatorConfig(
        metrics_calculators=[IoSMetricsCalculatorConfig()],
        triangle=7,
    ),
    matrix_formatter=MatrixFormatterConfig(
        output_type="dataframe",
        annotate=True,
    ),
    overlap_filter=ValueFilterConfig(
        ref_value=0.5,
        axis=0,
        operator=">",
        triangle=UPPER_ND,  # upper to align with containment hierarchies
        output="masked_values",
    ),
    overlap_aggregator=MatrixAggregatorConfig(
        operation="count",
        axis=1,
    ),
    spatial_scope=lobe_spatial_scope,
    # Spatial neighbor analysis is run post-hoc (see below).
    knn_analysis=None,
    rnn_analysis=None,
)


In [ ]:
@flow
def full_pipeline(img_path, scene_index=0, outdir=".", embedding_path="embeddings"):
    # load image
    img, metadata = ImageLoader(ilc).run(img_path)
    
    # TISSUE - brain lobes
    # preprocess
    tissue_img, tissue_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
    # segment tissue
    tissue_labels = TiledSegmentationFlow(tsfcfg).run(tissue_img, metadata=tissue_metadata, workers=2, verbose=0)
    if tissue_labels.ndim == img.ndim-2:
        print ("USING 2D TISSUE MASK")
        print(f"tissue_labels.ndim={tissue_labels.ndim}, np.max(tissue_labels)={np.max(tissue_labels)}, tissue_labels.dtype={tissue_labels.dtype}, img.ndim={img.ndim}")
        fc = FuncProcessorConfig(
            func="numpy.repeat",
            args=[img.shape[1]],
            kwargs={"axis":(0,)}, # stck along Z axis
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            output_dims={"Z": img.shape[-3], "Y": img.shape[-2], "X": img.shape[-1]},
            dtype=np.uint16,
        )
        arr = tissue_labels[np.newaxis, :, :]
        print (arr.shape)
        tissue_labels, tissue_metadata = FuncProcessor(fc).run(arr, metadata=tissue_metadata)
        print (f"{tissue_labels.shape}, {tissue_labels.dtype}, np.max(tissue_labels)={np.max(tissue_labels)}, {tissue_metadata}")
    else:
        print ("USING 3D TISSUE MASK")
    tissue_metadata = copy.deepcopy(tissue_metadata)
    tissue_metadata["channel_names"] = ["Lobe"]


    # BRAIN - all tissue combined
    #tissue_masks = labels_to_masks(tissue_labels)
    brain_mask = (tissue_labels>0).astype("uint16")
    focal_plane = len(tissue_labels)//2 
    brain_mask[focal_plane] = binary_dilation(brain_mask[focal_plane], iterations=1)
    brain_label = (brain_mask * 1).astype("uint16")
    brain_metadata = copy.deepcopy(tissue_metadata)

    # binary mask from segmented lobes
    #brain_mask = (tissue_labels > 0).astype("uint16")
    dcfg = FuncProcessorConfig(
        func="scipy.ndimage.binary_dilation",
        kwargs={"iterations": 1},  # XY pixels to grow
        iterator_config=ArrayIteratorConfig(slice_def=(-2, -1)),  # each YX plane
        normalize=False,
    )
    #brain_dilated, brain_metadata = FuncProcessor(dcfg).run(
    #    brain_mask,
    #    metadata=copy.deepcopy(tissue_metadata),
    #)
    #brain_label = brain_mask# brain_dilated#np.where(brain_dilated, 65535, 0).astype("uint16")
    brain_metadata["channel_names"] = ["Brain"]
    
    # CELLS
    # preprocess
    preprocessed, preprocessed_metadata = PreprocessFlow(cell_ppcfg).run(img, metadata=metadata)
    # split channels
    channels, channel_metadata = unstack_image(preprocessed, preprocessed_metadata, axis=metadata["channel_axis"], strict=False)
    # segment each channel separately
    cell_labels = SegmentationFlow(cell_sfcfg).mapped_run(channels, metadata=channel_metadata, workers=1)
    # cell_labels = [SegmentationFlow(cell_sfcfg).run(ch, metadata=ch_meta) for ch, ch_meta in zip(channels, channel_metadata)] 
    
    # Analyze CELLS and TISSUE
    # analyze regions in each channel separately
    combined_labels = [brain_label, tissue_labels, *cell_labels]
    combined_metadata = [brain_metadata, tissue_metadata, *channel_metadata]
    measurements = AnalysisFlow(acfg).run(combined_labels, metadata=combined_metadata)

    # save labels
    if outdir is None:
        outdir = Path(img_path).parent
    fname_stem = Path(img_path).stem
    imc = ImageWriterConfig(overwrite=True)
    outpaths = [os.path.join(outdir, f'{fname_stem}.scene-{meta.get("scene_index","")}.tif') for meta in combined_metadata]
    # print (outpaths)
    ImageWriter(imc).run.map(combined_labels, outpaths, metadata=combined_metadata)
    
    # make sure to resolve the futures to results
    return (
        resolve_futures(combined_labels),
        resolve_futures(combined_metadata),
        resolve_futures(measurements),
    )


In [ ]:
combined_labels, combined_metadata, measurements = full_pipeline(path, scene_index=scene_index, outdir=".", embedding_path=embedding_path)

In [ ]:
stackview.slice(np.concatenate(combined_labels, axis=-1))

In [ ]:
#for k,v in measurements.items():
#    print (k, type(v))

## Post-hoc homotypic / heterotypic kNN and RNN

Run once on the containment graph from `AnalysisFlow`. Distances are shared across all four analyses.

**THE FOLLOWING IS STILL TEMPORARY UNTIL WE GET A MORE GENERAL IMPLEMENTATION IN PLACE**

In [ ]:
from vistiq.graph import GraphFormatterConfig, resolve_subtree_origins, subtree_origin_key


def run_spatial_posthoc(graph, metadata, acfg, knn_cfg, rnn_cfg):
    """Homotypic + heterotypic kNN/RNN on an existing containment graph."""
    scope = acfg.spatial_scope
    origins = resolve_subtree_origins(
        graph,
        match=scope.match,
        exclude=scope.exclude,
        auto_root=scope.auto_root,
    )
    axes = tuple(metadata[0].get("axes", ())) if metadata else None
    dist_runner = KnnAnalysis(knn_cfg.model_copy(update={"mode": "global"}))

    spatial_results = {}
    for origin in origins:
        origin_key = subtree_origin_key(origin)
        dist = dist_runner.run(graph, node=origin, axes=axes).distance_matrix
        for mode in ("homotypic", "heterotypic"):
            spatial_results[f"knn_{mode}@{origin_key}"] = KnnAnalysis(
                knn_cfg.model_copy(update={"mode": mode})
            ).run(graph, node=origin, distance_matrix=dist, axes=axes)
            spatial_results[f"rnn_{mode}@{origin_key}"] = RnnAnalysis(
                rnn_cfg.model_copy(update={"mode": mode})
            ).run(graph, node=origin, distance_matrix=dist, axes=axes)
    return spatial_results, origins


def spatial_neighbor_summaries(spatial_results, acfg, knn_cfg, rnn_cfg, origins):
    gqcfg = acfg.spatial_graph_query
    index = (acfg.graph_formatter or GraphFormatterConfig()).index
    gqfmt_cfg = GraphQueryFormatterConfig(output_index=index)
    gqfmt = GraphQueryFormatter(gqfmt_cfg)
    frames = []
    for origin in origins:
        origin_key = subtree_origin_key(origin)
        for mode in ("homotypic", "heterotypic"):
            for analysis_type, result_key, k, radius in (
                ("knn", f"knn_{mode}@{origin_key}", knn_cfg.k, None),
                ("rnn", f"rnn_{mode}@{origin_key}", None, rnn_cfg.radius),
            ):
                result = spatial_results[result_key]
                gq = GraphQuery(
                    gqcfg.model_copy(
                        update={
                            "group_attribute": knn_cfg.grouping_attribute,
                            "neighbor_analysis": analysis_type,
                            "neighbor_k": k,
                            "neighbor_radius": radius,
                        }
                    )
                )
                frame = gqfmt.run(gq.run(result.graph), attribute="neighbor_summary")
                prefix = f"{analysis_type}_{mode}"
                if len(origins) > 1:
                    prefix = f"{prefix}_{origin_key}"
                frames.append(frame.add_prefix(f"{prefix}__"))
    if not frames:
        return pd.DataFrame()
    summary = pd.concat(frames, axis=1)
    return summary.loc[:, ~summary.columns.duplicated()]


spatial_results, spatial_origins = run_spatial_posthoc(
    measurements["containment_graph"],
    combined_metadata,
    acfg,
    knn_spatial_cfg,
    rnn_spatial_cfg,
)
spatial_summary = spatial_neighbor_summaries(
    spatial_results,
    acfg,
    knn_spatial_cfg,
    rnn_spatial_cfg,
    spatial_origins,
)

measurements["spatial_posthoc"] = spatial_results
measurements["spatial_summary"] = spatial_summary
print(f"spatial origins: {[subtree_origin_key(o) for o in spatial_origins]}")
print(f"spatial summary columns: {len(spatial_summary.columns)}")
spatial_summary.head()


# Query graph for object ancestor lineage

1. Subcellular cellular Dpn -> tissue lobe -> organ brain
2. Count descendants in each channel

In [ ]:
dag = measurements["containment_graph"]

gqcfg = GraphQueryConfig(
    attributes=["descendant_counts", "ancestor_lineage"],
    filter_attribute="channel",
    filter_value="Lobe",
    include_attributes=["object_name", "label", "channel", "volume"],
    lineage_value_attribute="label",
)
gq = GraphQuery(gqcfg)
gqfmt = GraphQueryFormatter(GraphQueryFormatterConfig())
result = gq.run(dag, node=None)
df_counts = gqfmt.run(result, attribute="descendant_counts")
df_lineage = gqfmt.run(result, attribute="ancestor_lineage")
df = pd.concat([df_counts, df_lineage], axis=1)
df = df.loc[:, ~df.columns.duplicated()].sort_values(["label"])
df

In [ ]:
dag = measurements["containment_graph"]
# Any Lobe nodes left?
lobe_nodes = [
    n for n, attrs in dag.nodes(data=True)
    if attrs.get("channel") == "Lobe"
]
print(f"Lobe nodes in graph: {len(lobe_nodes)}")
print(f"Total graph nodes: {dag.number_of_nodes()}")
result = gq.run(dag, node=None)
print(f"Seed rows: {len(result['descendant_counts'])}")

In [ ]:
# add binary +/- categorization for channel overlap (based on channel overlap count) 
features = measurements["region_analyzer_all"].join(spatial_summary, how="left").sort_values(["channel", "label"])
for ch in np.unique(features["channel"]):
    count_col = f"count {ch}"
    if count_col in features.columns.values:
        features[f"{ch} +"] = features[count_col]>0
#features.columns.values

In [ ]:
# reorder columns
cols = list(features.columns.values)
basics = ["label", "object_name", "channel","lineage Lobe"] 
positives = [c for c in cols if "+" in c] 
counts = [c for c in cols if "count" in c]
lineage = []#[c for c in cols if "lineage" in c]
rest = [c for c in cols if c not in (basics+positives+counts+lineage)]
reordered_cols = basics + counts + positives + lineage + rest
features = features[reordered_cols]
features[basics+positives].groupby(["lineage Lobe", "channel","EdU +"]).count()

#features[["channel", "object_name", "lineage Lobe"]].groupby(["lineage Lobe", "channel"]).count()
#features[(features["channel"]=="Dpn") & (~features["count EdU"].isna())]["count EdU"]

## Save analysis bundle

Persists features, spatial matrices/graphs, and the containment DAG so napari layers can be rebuilt without rerunning `full_pipeline`.


In [ ]:
analysis_bundle_dir = save_analysis(
    Path(path).parent,
    stem=Path(path).stem,
    measurements=measurements,
    features=features,
    spatial_results=spatial_results,
    spatial_origins=spatial_origins,
    knn_cfg=knn_spatial_cfg,
    rnn_cfg=rnn_spatial_cfg,
)
print(analysis_bundle_dir)
